In [38]:
from spin_lattices import KagomeLattice, SpinLattice, SquareLattice, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import torch
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm
from torch_geometric.nn import GCNConv, global_add_pool, GATConv, global_mean_pool
import torch.nn.functional as F
from torch_geometric.data import DataLoader
from torch.nn.functional import mse_loss
from torch.optim import Adam
import numpy as np

In [55]:
lat = KagomeLattice(width=2, height=2)

In [56]:
system = HeisenbergJ1J2(
    lattice=lat,
    J1=1.0,
    J2=0.8,
    use_symmetries=True,
    spin_inversion=1,
    skip_symmetries_whitelist=True,
    ground_state_cache_dir=Path("groundstates")
)

2023-06-20 12:43:50.459 | WARNING  | heisenberg_hamiltonians:__init__:414 - Symmetries are not tested with KagomeLattice2x2, and can produce incorrect results. Using them anyway due to skip_symmetries_whitelist=True.
2023-06-20 12:43:50.461 | DEBUG    | heisenberg_hamiltonians:__init__:446 - number_spins=12
2023-06-20 12:43:50.467 | DEBUG    | heisenberg_hamiltonians:__init__:456 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-06-20 12:43:50.500 | DEBUG    | heisenberg_hamiltonians:__init__:465 - Hilbert space dimension is 47
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


In [58]:
df = (
    system.get_df_ground_state(unpack_configurations=True, expand_basis_columns=True)
    .drop("eigenstate_coeff", axis=1)
#    .sample(n=50000)
    .assign(target=lambda df: np.log(df["amplitude"]))
    .drop("amplitude", axis=1)
)

In [59]:
# Get edge list
edge_list = torch.tensor(lat.as_igraph().get_edgelist(), dtype=torch.long).t().contiguous()

data_list = []

for i, row in tqdm(list(df.iterrows())):
    # Get node features
    node_features = torch.tensor(row.drop('target').values, dtype=torch.float).view(-1, 1)
    # Get node target
    node_target = torch.tensor([row['target']], dtype=torch.float)
    
    data = Data(x=node_features, edge_index=edge_list, y=node_target)
    data_list.append(data)


  0%|          | 0/924 [00:00<?, ?it/s]

In [66]:
class Net(torch.nn.Module):
    def __init__(self, num_node_features):
        super(Net, self).__init__()
        self.conv1 = GCNConv(num_node_features, 16)
        self.conv2 = GCNConv(16, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        x = global_mean_pool(x, batch)

        return x

In [67]:
num_node_features = data_list[0].num_features  # Assuming all Data objects have the same number of features
model = Net(num_node_features)

In [68]:
def overlap(x, y):
    return torch.dot(x, y) / (torch.norm(x) * torch.norm(y))

In [71]:
# Create a DataLoader
loader = DataLoader(data_list, batch_size=32, shuffle=True)
full_data = Batch.from_data_list(data_list)

# Instantiate the model
num_node_features = data_list[0].num_features
model = Net(num_node_features)

# Use mean squared error loss for regression tasks
criterion = mse_loss

# Use Adam optimizer
optimizer = Adam(model.parameters(), lr=1e-2)

# Set device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Training loop
for epoch in range(200):
    total_loss = 0
    model.train()
    for data in loader:
        # Move data and target to the correct device
        data = data.to(device)
        target = data.y.to(device)

        # Forward pass
        out = model(data)
        loss = criterion(out, target.view(-1, 1))

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch: {epoch+1}, Loss: {total_loss/len(loader)}')
    
    # print(f"{overlap(model(full_data).view(-1), full_data.y).item()=}")

Epoch: 1, Loss: 13.894110153461325
Epoch: 2, Loss: 3.444392066577385
Epoch: 3, Loss: 1.7589437262765293
Epoch: 4, Loss: 1.7018099529989834
Epoch: 5, Loss: 1.668823412780104
Epoch: 6, Loss: 1.6409654144583077
Epoch: 7, Loss: 1.6096472329106823
Epoch: 8, Loss: 1.5836606683402226
Epoch: 9, Loss: 1.5595216689438656
Epoch: 10, Loss: 1.5438866574188759
Epoch: 11, Loss: 1.519935198898973
Epoch: 12, Loss: 1.4943343976448322
Epoch: 13, Loss: 1.4825742573573673
Epoch: 14, Loss: 1.4657146355201458
Epoch: 15, Loss: 1.4589820709721795
Epoch: 16, Loss: 1.4439214488555645
Epoch: 17, Loss: 1.4361282772031323
Epoch: 18, Loss: 1.4330469061588418
Epoch: 19, Loss: 1.4370324344470584
Epoch: 20, Loss: 1.4294309739408821
Epoch: 21, Loss: 1.41177250599039
Epoch: 22, Loss: 1.4112036310393234
Epoch: 23, Loss: 1.408619615538367
Epoch: 24, Loss: 1.4084663062260068
Epoch: 25, Loss: 1.4109544301855153
Epoch: 26, Loss: 1.4086772018465503
Epoch: 27, Loss: 1.4050987116221725
Epoch: 28, Loss: 1.410921939488115
Epoch: 2

In [73]:
overlap(torch.exp(model(full_data)).view(-1), torch.exp(full_data.y))

tensor(0.6655, grad_fn=<DivBackward0>)

In [234]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# df = pd.read_csv('your_file.csv')
# permutation = [2, 0, 1, 3, 4]
def show_permutation(permutation):
    fig, ax = plt.subplots()
    scat = ax.scatter(df['emb_x'], df['emb_y'])

    # Define the number of intermediate frames for each permutation
    num_inter_frames = 20

    # Function to interpolate between two points
    def interpolate_points(p1, p2, n_steps=num_inter_frames):
        ratio = np.linspace(0, 1, n_steps)
        x = p1[0] * (1-ratio) + p2[0] * ratio
        y = p1[1] * (1-ratio) + p2[1] * ratio
        return np.array([x, y]).T

    # Prepare the list to store all steps
    steps = []

    # Compute all steps from the initial positions to the final ones
    for i in range(len(df)):
        start = df.loc[i, ['emb_x', 'emb_y']].values
        end = df.loc[permutation[i], ['emb_x', 'emb_y']].values
        steps.append(interpolate_points(start, end))

    # Convert list of steps into an array
    steps = np.array(steps)

    # Function to update scatter
    def update(num):
        # Get new coordinates for points
        new_positions = steps[:, num % num_inter_frames, :]

        # Update scatter object
        scat.set_offsets(new_positions)

        return scat,

    # Compute total number of frames
    total_frames = num_inter_frames

    # Create animation using the update function
    ani = animation.FuncAnimation(fig, update, frames=range(total_frames), blit=True)

    return HTML(ani.to_html5_video())

In [261]:
[x for x in automorphisms if x[4] == 4]

[(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17), (6, 9, 8, 7, 4, 5, 0, 3, 2, 1, 16, 17, 12, 15, 14, 13, 10, 11)]